# Appendix simulation: cross fitting versus no cross fitting

Compatible loss-link pairs are run on the same three DGPs with `cross_fit=True` and `cross_fit=False`. Both ATE and ATT are estimated.

In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    parent = REPO_ROOT.parent
    if parent == REPO_ROOT:
        raise RuntimeError("Run this notebook from inside the genriesz repository.")
    REPO_ROOT = parent

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz import grr_ate, grr_att
from genriesz.basis import BaseBasis, TreatmentInteractionBasis
from genriesz.experiments import (
    COMPATIBLE_LOSSES,
    ESTIMANDS,
    ESTIMATORS_ALL,
    RANDOM_SEED,
    TREATMENT_INDEX,
    CoverageDiagnosticBasis,
    SelectedColumnsBasis,
    fit_grr_love_plot_data,
    fit_matching_ate,
    fit_one_grr,
    fit_one_grr_with_basis,
    fit_one_incompatible,
    fit_one_plugin_logistic,
    generator_shift_for_estimand,
    load_ihdp_replication,
    load_lalonde,
    make_basis,
    make_compatible_generator,
    make_coverage_diagnostic_data,
    make_dimension_data,
    make_kang_schafer_data,
    make_kernel_gp_data,
    make_score_guided_data,
    make_simulation_data,
    result_to_rows,
    summarize_estimates,
    true_theta,
)

DATA_DIR = REPO_ROOT / "notebooks" / "experiments" / "data"
TABLE_CONFIG = {"float_format": "{:.4f}", "max_rows": 200}
PLOT_CONFIG = {
    "figure_size": (9.0, 5.2),
    "figure_size_wide": (12.0, 5.2),
    "title_fontsize": 14,
    "axis_fontsize": 12,
    "tick_fontsize": 10,
    "legend_fontsize": 10,
    "line_width": 2.0,
    "marker_size": 5,
    "box_width": 0.70,
    "grid_alpha": 0.30,
    "dpi": 140,
    "squared_error_y_scale": "log",
    "squared_error_floor": 1e-12,
}
METHOD_LABELS = {
    "ra": "RA",
    "rw": "RW (IPW)",
    "arw": "ARW (AIPW)",
    "tmle": "TMLE",
    "SQ": "SQ-Riesz",
    "UKL": "UKL-Riesz",
    "BKL": "BKL-Riesz",
    "BP(0.5)": "BP-Riesz (omega = 0.5)",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random forest leaves",
    "rff": "Random Fourier features",
    "matching": "Nearest-neighbor matching",
}
METHOD_COLORS = {
    "SQ": "#4C78A8",
    "UKL": "#F58518",
    "BKL": "#54A24B",
    "BP(0.5)": "#B279A2",
    "rkhs": "#4C78A8",
    "polynomial": "#F58518",
    "rf": "#54A24B",
    "rff": "#E45756",
    "matching": "#72B7B2",
}
DISPLAY_LABELS = {
    "ra": "RA",
    "rw": "RW",
    "arw": "ARW",
    "tmle": "TMLE",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random Forest",
    "rff": "Random Fourier Features",
    "matching": "Nearest-Neighbor Matching",
    "regressor": "Regressor",
    "covariate": "Covariate",
}
LABEL_COLUMNS = ("estimator", "basis", "basis_mode", "loss", "loss_link_pair")
pd.options.display.max_rows = TABLE_CONFIG["max_rows"]


def label_of(value):
    """Return the display label for a stored result key."""

    return DISPLAY_LABELS.get(str(value), str(value))


def prettify_method(text):
    """Replace stored result keys inside a composite display label."""

    out = str(text)
    for key, value in DISPLAY_LABELS.items():
        out = re.sub(r"(?<![A-Za-z0-9_])" + re.escape(key) + r"(?![A-Za-z0-9_])", value, out)
    return out


def prettify_labels(frame):
    """Return a copy with known result-key columns formatted for display."""

    out = frame.copy()
    for column in LABEL_COLUMNS:
        if column in out.columns:
            out[column] = out[column].map(label_of)
    return out


def display_table(frame, *, caption=None, digits=4):
    """Display a rounded table without changing the stored results."""

    table = prettify_labels(frame)
    numeric_columns = table.select_dtypes(include=[np.number]).columns
    table[numeric_columns] = table[numeric_columns].round(digits)
    styler = table.style.format(precision=digits)
    if caption is not None:
        styler = styler.set_caption(caption)
    display(styler)


In [ ]:
N_REPLICATIONS = 100
SAMPLE_SIZE = 1200
DGP_NAMES = ["DGP 1: smooth heterogeneous effects", "DGP 2: weak overlap nonlinear design", "DGP 3: high-dimensional sparse confounding"]
BASIS_KIND = "rkhs"
RIEZ_FEATURES = 120
RIEZ_LAMBDA = 3e-1
FOLDS = 5

In [ ]:
rows = []
for dgp_name in DGP_NAMES:
    for rep in range(N_REPLICATIONS):
        data = make_simulation_data(dgp_name, n=SAMPLE_SIZE, seed=400000 + 1009 * rep)
        for cross_fit in [True, False]:
            for loss_spec in COMPATIBLE_LOSSES:
                for estimand in ESTIMANDS:
                    fit_rows = fit_one_grr(
                        data,
                        estimand=estimand,
                        loss_spec=loss_spec,
                        basis_kind=BASIS_KIND,
                        basis_mode="regressor",
                        cross_fit=cross_fit,
                        lam=RIEZ_LAMBDA,
                        basis_features=RIEZ_FEATURES,
                        folds=FOLDS,
                        estimators=ESTIMATORS_ALL,
                        random_state=rep,
                    )
                    for row in fit_rows:
                        row["dgp"] = dgp_name
                        row["replication"] = rep
                    rows.extend(fit_rows)
crossfit_results = pd.DataFrame(rows)
crossfit_summary = summarize_estimates(crossfit_results, ["dgp", "estimand", "cross_fit", "loss", "estimator"])

for estimand_name in ESTIMANDS:
    table_df = crossfit_summary[crossfit_summary["estimand"] == estimand_name].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["dgp", "cross_fit", "loss", "estimator"])
    display_table(table_df, caption=f"Appendix cross fitting comparison: {estimand_name}")


In [ ]:
# Box plots of ARW squared error by cross-fitting status and loss. Separate figures by estimand and DGP.
plot_df = crossfit_results[(crossfit_results["status"] == "ok") & (crossfit_results["estimator"] == "arw")].copy()
plot_df["method"] = plot_df["loss"] + " | cf=" + plot_df["cross_fit"].astype(str)
plot_df["squared_error_plot"] = plot_df["squared_error"].clip(lower=PLOT_CONFIG["squared_error_floor"])

for estimand_name in ESTIMANDS:
    for dgp_name in DGP_NAMES:
        panel_df = plot_df[(plot_df["estimand"] == estimand_name) & (plot_df["dgp"] == dgp_name)].copy()
        if panel_df.empty:
            print(f"No plot data for {estimand_name}, {dgp_name}.")
            continue
        fig, ax = plt.subplots(figsize=PLOT_CONFIG["figure_size_wide"], dpi=PLOT_CONFIG["dpi"])
        ordered_methods = sorted(panel_df["method"].unique())
        box_data = [panel_df.loc[panel_df["method"] == m, "squared_error_plot"].dropna().to_numpy() for m in ordered_methods]
        box = ax.boxplot(box_data, tick_labels=[prettify_method(_m) for _m in ordered_methods], widths=PLOT_CONFIG["box_width"], patch_artist=True, showfliers=False)
        for patch, method in zip(box["boxes"], ordered_methods):
            loss_name = method.split(" | ")[0]
            patch.set_facecolor(METHOD_COLORS.get(loss_name, "#CCCCCC"))
            patch.set_alpha(0.75)
        ax.set_yscale(PLOT_CONFIG["squared_error_y_scale"])
        ax.set_title(f"{estimand_name}: cross fitting comparison, {dgp_name}", fontsize=PLOT_CONFIG["title_fontsize"])
        ax.set_xlabel("Loss and cross-fitting status", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.set_ylabel("Squared error", fontsize=PLOT_CONFIG["axis_fontsize"])
        ax.tick_params(axis="x", labelrotation=45, labelsize=PLOT_CONFIG["tick_fontsize"])
        ax.grid(axis="y", alpha=PLOT_CONFIG["grid_alpha"])
        fig.tight_layout()
        plt.show()
